In [ ]:
import os, sys
import logging
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from hmmlearn.hmm import GaussianHMM
from multiprocessing import Pool, cpu_count
import pandas_ta as ta

sys.path.append(os.path.dirname(os.path.realpath('__file__')) + '/../../')
from src.common.constants import GlobalConstants

# Constants
TRAIN_START = '2020-01-01'
TRAIN_END = '2022-01-01'
TEST_START = '2022-01-02'
TEST_END = '2023-01-01'
REBALANCE_FREQ = 30  # days
TRANSACTION_COST = 0.001  # 0.1% per trade
ROLLING_WINDOW = 252 * 2  # ~2 years
MIN_TREND_DURATION = 5  # Minimum days in trending state

current_date = datetime.now().strftime('%Y%m%d')
log_file_name = GlobalConstants.strategy_log_path + os.path.splitext(os.path.basename('__file__'))[0] + '_' + current_date + '.log'

logging.basicConfig(filename=log_file_name,
            level=logging.INFO,
            format=GlobalConstants.strategy_log_format)

# Update the default date/time format
logging.Formatter.default_time_format = '%Y-%m-%d %H:%M:%S'
logging.Formatter.default_msec_format = '%s.%03d'

#########################
# Data Loading & Features
#########################

def load_stock_data(symbol, data_dir='data'):
    file_path = os.path.join(data_dir, f"{symbol}.csv")
    try:
        df = pd.read_csv(file_path, parse_dates=['Date'])
        df.sort_values('Date', inplace=True)
        # Remove timezone information if present:
        df['Date'] = df['Date'].dt.tz_localize(None)
        
        if df.empty or df['Close'].isna().all():
            logging.info(f"Invalid data for {symbol}.")
            return None
        return df
    except Exception as e:
        logging.info(f"Error loading data for {symbol}: {e}")
        return None

def compute_features(df):
    df['log_price'] = np.log(df['Close'])
    df['log_return'] = df['log_price'].diff().where(df['Volume'] > 0, np.nan)
    df['volatility'] = df['log_return'].rolling(window=20).std()
    df = compute_adx_feature(df)
    df = compute_moving_average_slope(df)
    df = compute_rsi_feature(df)
    df = compute_macd_feature(df)
    df.dropna(inplace=True)
    return df

def compute_adx_feature(df, window=14):
    if not all(col in df.columns for col in ['High', 'Low', 'Close']):
        df['adx'] = np.NaN
        return df
    try:
        adx = ta.adx(df['High'], df['Low'], df['Close'], length=window)
        # extract just the ADX values and add to the dataframe - adx above have following columns : 'ADX_14', 'DMP_14', 'DMN_14'
        df['adx'] = adx[f'ADX_{window}']

    except Exception as e:
        logging.info(f"Error computing ADX: {e}")
        df['adx'] = np.NaN
    return df

def compute_moving_average_slope(df, window=20):
    try:
        df['ma'] = df['Close'].rolling(window=window, min_periods=window).mean()
        df['ma_slope'] = df['ma'].diff()
    except Exception as e:
        logging.info(f"Error computing MA slope: {e}")
        df['ma'] = np.NaN
        df['ma_slope'] = np.NaN
    return df

def compute_rsi_feature(df, window=14):
    try:
        df['rsi'] = ta.rsi(df['Close'], length=window)
    except Exception as e:
        logging.info(f"Error computing RSI: {e}")
        df['rsi'] = np.NaN
    return df

def compute_macd_feature(df, window_slow=26, window_fast=12, window_sign=9):
    try:
        macd = ta.macd(df['Close'], fast=window_fast, slow=window_slow, signal=window_sign)
        df['macd_diff'] = macd[f'MACDh_{window_fast}_{window_slow}_{window_sign}']
    except Exception as e:
        logging.info(f"Error computing MACD: {e}")
        df['macd_diff'] = np.NaN
    return df

#########################
# HMM Tuning
#########################

def compute_n_params(model, n_features):
    m = model.n_components
    d = n_features
    if model.covariance_type == 'full':
        cov_params = m * d * d
    elif model.covariance_type == 'diag':
        cov_params = m * d
    elif model.covariance_type == 'spherical':
        cov_params = m * 1
    else:
        raise ValueError("Unsupported covariance type")
    total_params = (m - 1) + m * (m - 1) + m * d + cov_params
    return total_params

def compute_bic(model, X):
    n_samples = len(X)
    logL = model.score(X)
    n_params = compute_n_params(model, X.shape[1])
    return -2 * logL + n_params * np.log(n_samples)

def tune_hmm_state(X, state_range=range(2, 7), covariance_types=['full', 'diagonal'], max_iter=1000):
    best_bic = np.inf
    best_config = None
    best_model = None
    n_features = X.shape[1] if X.ndim > 1 else 1
    for n_states in state_range:
        for cov_type in covariance_types:
            try:
                model = GaussianHMM(
                    n_components=n_states,
                    covariance_type=cov_type,
                    n_iter=max_iter,
                    random_state=42,
                    tol=1e-3
                )
                model.fit(X)
                if not model.monitor_.converged:
                    logging.info(f"HMM ({n_states} states, {cov_type}) did not converge.")
                    continue
                else:
                    logging.info(f"HMM ({n_states} states, {cov_type}) converged after {model.monitor_.iter} iterations.")
                    
                bic = compute_bic(model, X)
                logging.info(f"HMM: {n_states} states, {cov_type}, BIC={bic}")
                if bic < best_bic:
                    best_bic = bic
                    best_config = (n_states, cov_type)
                    best_model = model
            except Exception as e:
                logging.info(f"Error tuning HMM ({n_states}, {cov_type}): {e}")
    if best_model is None:
        logging.info("No valid HMM model found.")
        
    logging.info(f"Best HMM: {best_config}, BIC={best_bic}, Best Model: {best_model}")
    return best_config, best_model

#########################
# Regime Detection
#########################

def apply_viterbi(df, model):
    X = df[['log_return', 'adx', 'ma_slope', 'rsi', 'macd_diff']].values
    states = model.predict(X)
    df['regime'] = states
    logging.info(f"Regime distribution: {df['regime'].value_counts(normalize=True)}")
    return df

def identify_trending_regime(df, model):
    states = model.predict(df[['log_return', 'adx', 'ma_slope', 'rsi', 'macd_diff']].values)
    df['regime'] = states
    current_regime = df['regime'].iloc[-1]
    regime_means = {}
    for state in range(model.n_components):
        state_df = df[df['regime'] == state]
        regime_means[state] = {
            'log_return': state_df['log_return'].mean(),
            'adx': state_df['adx'].mean(),
            'rsi': state_df['rsi'].mean(),
            'macd_diff': state_df['macd_diff'].mean()
        }
    trending_criteria = {
        'log_return': lambda x: x > 0,
        'adx': lambda x: x > 25,
        'rsi': lambda x: x > 50,
        'macd_diff': lambda x: x > 0
    }
    is_trending = all(trending_criteria[feature](regime_means[current_regime][feature]) for feature in trending_criteria)
    recent_states = df['regime'].tail(MIN_TREND_DURATION)
    persistent = (recent_states == current_regime).all()
    return is_trending and persistent, current_regime, regime_means

#########################
# Ranking & Backtesting
#########################

def process_stock_for_date(args):
    symbol, cutoff_date, params, data_dir = args
    df = load_stock_data(symbol, data_dir)
    if df is None or len(df) < ROLLING_WINDOW:
        return symbol, None, None
    df = df[df['Date'] <= cutoff_date].tail(ROLLING_WINDOW)
    df = compute_features(df)
    if df.empty:
        return symbol, None, None
    X = df[['log_return', 'adx', 'ma_slope', 'rsi', 'macd_diff']].values
    best_config, best_model = tune_hmm_state(X)
    if best_model is None:
        return symbol, None, None
    df = apply_viterbi(df, best_model)
    is_trending, current_regime, regimes = identify_trending_regime(df, best_model)
    if not is_trending:
        return symbol, 0, df['Close'].iloc[-1]
    base_score = regimes[current_regime]['log_return']
    recent_period = df.tail(20)
    avg_ma_slope = recent_period['ma_slope'].mean() or 0
    avg_adx = recent_period['adx'].mean() or 0
    score = base_score + params['weight_ma'] * avg_ma_slope + params['weight_adx'] * avg_adx
    return symbol, score, df['Close'].iloc[-1]

def backtest_portfolio(nifty50_symbols, params, data_dir, start_date, end_date, rebalance_freq):
    dates = pd.date_range(start_date, end_date, freq=f'{rebalance_freq}D')
    portfolio_returns = []
    portfolio_dates = []
    for i in range(len(dates) - 1):
        rebalance_date = dates[i]
        next_date = dates[i + 1]
        with Pool(cpu_count()) as pool:
            results = pool.map(process_stock_for_date, [(symbol, rebalance_date, params, data_dir) for symbol in nifty50_symbols])
        scores = {sym: score for sym, score, _ in results if score is not None}
        prices = {sym: price for sym, _, price in results if price is not None}
        if not scores:
            continue
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        top_symbols = [x[0] for x in ranked[:5]]
        logging.info(f"At {rebalance_date.date()}: Top stocks: {top_symbols}")
        returns = []
        for symbol in top_symbols:
            df = load_stock_data(symbol, data_dir)
            if df is None:
                continue
            df = df[(df['Date'] >= rebalance_date) & (df['Date'] <= next_date)]
            if df.empty or len(df) < 2:
                continue
            price_next = df['Close'].iloc[-1]
            price_rebalance = prices[symbol]
            ret = (price_next - price_rebalance) / price_rebalance - TRANSACTION_COST
            returns.append(ret)
        if returns:
            period_return = np.mean(returns)
            portfolio_returns.append(period_return)
            portfolio_dates.append(next_date)
    portfolio_returns = pd.Series(portfolio_returns, index=portfolio_dates)
    cumulative_return = (1 + portfolio_returns).prod() - 1
    daily_returns = portfolio_returns / rebalance_freq
    sharpe_ratio = np.sqrt(252) * daily_returns.mean() / daily_returns.std() if daily_returns.std() > 0 else np.NaN
    return portfolio_returns, {'cumulative_return': cumulative_return, 'sharpe_ratio': sharpe_ratio}

def tune_parameters(nifty50_symbols, data_dir, train_start, train_end):
    candidate_weight_ma = [0.0005, 0.001, 0.0015]
    candidate_weight_adx = [0.005, 0.01, 0.015]
    best_sharpe = -np.inf
    best_params = None
    for w_ma in candidate_weight_ma:
        for w_adx in candidate_weight_adx:
            params = {'weight_ma': w_ma, 'weight_adx': w_adx}
            _, perf = backtest_portfolio(nifty50_symbols, params, data_dir, train_start, train_end, REBALANCE_FREQ)
            if perf['sharpe_ratio'] > best_sharpe:
                best_sharpe = perf['sharpe_ratio']
                best_params = params
                logging.info(f"New best params: {params}, Sharpe: {best_sharpe}")
    return best_params

def main():
    # nifty50_symbols = [
    #     'RELIANCE', 'TCS', 'HDFCBANK', 'INFY', 'ICICIBANK', 'HINDUNILVR', 'KOTAKBANK', 'SBIN',
    #     'BAJFINANCE', 'BHARTIARTL', 'ITC', 'HCLTECH', 'ASIANPAINT', 'LT', 'AXISBANK', 'MARUTI',
    #     'SUNPHARMA', 'TITAN', 'ULTRACEMCO', 'DRREDDY', 'WIPRO', 'NTPC', 'POWERGRID', 'M&M',
    #     'HINDALCO', 'BPCL', 'COALINDIA', 'GRASIM', 'JSWSTEEL', 'ADANIPORTS', 'TATASTEEL',
    #     'UPL', 'BRITANNIA', 'NESTLEIND', 'DIVISLAB', 'ONGC', 'SBILIFE', 'TECHM', 'SHREECEM',
    #     'GAIL', 'EICHERMOT', 'ICICIPRULI', 'TATAMOTORS', 'INDUSINDBK', 'CIPLA', 'VEDL',
    #     'BIOCON', 'COLPAL', 'DMART'
    # ]
    nifty50_symbols = ['RELIANCE']
    data_dir = '/home/qa/runtime/data/analysis_out/garch'
    # commented for now as we have got best param after one full run 
    # best_params = {'weight_ma': 0.0005, 'weight_adx': 0.005}
    best_params = tune_parameters(nifty50_symbols, data_dir, TRAIN_START, TRAIN_END)
    
    if best_params:
        logging.info(f"Best Parameters: {best_params}")
        _, test_perf = backtest_portfolio(nifty50_symbols, best_params, data_dir, TEST_START, TEST_END, REBALANCE_FREQ)
        logging.info(f"Test Performance: {test_perf}")
    else:
        logging.info("Parameter tuning failed.")

if __name__ == '__main__':
    main()

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
from hmmlearn.hmm import GaussianHMM

# Load stock price data (e.g., Apple from 2020-2023)
stock_symbol = 'RELIANCE.NS'
data = yf.download(stock_symbol, start='2020-01-01', end='2023-01-01')

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
from hmmlearn.hmm import GaussianHMM

# Load st ock price data (e.g., Apple from 2020-2023)
# stock_symbol = 'RELIANCE.NS'
# data = yf.download(stock_symbol, start='2020-01-01', end='2023-01-01')
# data = data[['Close']]  # Keep only closing prices

# Compute log returns (difference in log prices)
data['log_return'] = np.log(data['Close'] / data['Close'].shift(1))
data['volatility'] = data['log_return'].rolling(window=20).std()
data = compute_adx_feature(data)
data = compute_moving_average_slope(data)
data = compute_rsi_feature(data)
data = compute_macd_feature(data)
data.dropna(inplace=True)  # Remove NaN values

# Prepare data for HMM (log returns as the feature)
# X = data[['log_return']].values
X = data[['log_return', 'adx', 'ma_slope', 'rsi', 'macd_diff']].values

# Train a simple HMM with 3 states
model = GaussianHMM(n_components=3, covariance_type="full", n_iter=1000, random_state=42)
model.fit(X)

# Predict the regimes (hidden states)
hidden_states = model.predict(X)

# Add regimes to the dataframe
data['regime'] = hidden_states

# Visualize the results
plt.figure(figsize=(14, 7))
for regime in sorted(data['regime'].unique()):
    regime_data = data[data['regime'] == regime]
    plt.plot(regime_data.index, regime_data['Close'], '.', label=f"Regime {regime}")

# Plot the closing prices as a reference line
plt.plot(data.index, data['Close'], color='black', alpha=0.3, label='Close Price')
plt.title(f"{stock_symbol} Price with HMM Regime Classification")
plt.xlabel("Date")
plt.ylabel("Close Price")
plt.legend()
plt.show()